# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 17 · Attribute historical response value

**Same data, same trees, controlled history features.**

The support-only arm contains all counts, cold-start indicators and recency fields used by both outcome-derived arms. It distinguishes behavior from frequency. Maximum: three arms × two coordinates × two folds = 12 new coordinate models. Preserved controls are replayed, not refitted. No ensemble or model search.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round7')
OUT = Path('/home/sagemaker-user/nfl-feature-round7-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract this kit first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=2):
    cmd = [str(PY), str(KIT/'run_round.py'), stage, '--fold', str(fold)]
    env = os.environ.copy()
    env.update(OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
    process = subprocess.Popen(cmd, cwd=str(KIT), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        process.wait(timeout=20)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped with exit {code}. Keep checkpoints and return the generated report; do not change settings.')
def read(name):
    return json.loads((OUT/name).read_text())
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## 1. Later fold 2
Six small coordinate fits, checkpointed every 30 boosting iterations. The fixed model sees the original 72 columns plus 9, 15 or 21 historical fields. Screening uses training rows only.

In [ ]:
run('fit', fold=2)
r = read('fold_2/summary.json')
assert r['status'] == 'history_fold_complete'
print(json.dumps({'metrics':r['metrics'],'retained_columns':r['retained_columns'],'new_coordinate_models':r['new_coordinate_models']}, indent=2))

## 2. Later fold 3 — fixed cost guard
Proceed unchanged only through this cell. If both outcome-derived arms are at least 5% worse than the preserved control on fold 2, the worker records a futility stop and does not fit fold 3. Do not bypass it. Smaller mixed effects still proceed so unfavorable folds are not hidden.

In [ ]:
run('fit', fold=3)
if (OUT/'fold_3/futility.json').exists():
    print(json.dumps(read('fold_3/futility.json'), indent=2))
else:
    r = read('fold_3/summary.json')
    print(json.dumps({'status':r['status'],'metrics':r['metrics'],'new_coordinate_models':r['new_coordinate_models']}, indent=2))

## 3. Fresh-process replay of encoders and models
Reconstruct ordered training features and frozen evaluation features exactly; independently reload completed models. Replay cannot fit a missing scientific model. Historical-encoder reconstruction is not a model refit.

In [ ]:
run('replay')
r = read('replay.json')
assert r['status'] == 'history_replay_exact' and r['new_coordinate_models'] == 0
print(json.dumps(r, indent=2))

## 4. Inspect all comparisons
A role history must beat both the original control and support-only arm. Player history must additionally beat role history. Require at least 1% pooled improvement, a negative adjusted upper paired-game difference bound, and improvement in both folds for every required contrast. These are reused-game screens; multiplicity adjustment within this round does not correct all prior adaptive research.

In [ ]:
run('summarize')
r = read('summary.json')
print(json.dumps({'status':r['status'],'pooled_metrics':r['pooled_metrics'],
    'role_history_earned_further_testing':r['role_history_earned_further_testing'],
    'player_history_earned_further_testing':r['player_history_earned_further_testing']}, indent=2))
show(visuals.current_metrics(OUT), 'history_fold_metrics')
show(visuals.current_intervals(OUT), 'history_contrasts')
show(visuals.horizon_errors(OUT), 'history_horizon_errors')
show(visuals.cold_start_errors(OUT), 'history_cold_start_errors')

## 5. Return the aggregate evidence
The return ZIP excludes historical tables, IDs, per-row targets/predictions, feature arrays and model weights. Save this notebook before stopping the existing space. Do not publish private notebook outputs or use `git add .`.

In [ ]:
run('report')
print(OUT/'nfl_feature_round7_report.zip')

## Decision boundary
A positive screen earns a separately versioned integration into a replayable temporal forecaster and fresh validation—not a Kaggle submission. A failed screen stops this historical encoder unchanged. No additional bins, role switches, mixtures, or hyperparameter changes are authorized by this notebook.